# DelayLens: run the reviewed project

Upload the complete project ZIP if using Colab. Use Python 3.12 for the verified dependency lock. Run all cells. The original dashboard model is a frozen historical demonstration, not the preferred research method.

In [1]:
import sys, os, json, subprocess, tempfile, zipfile
from pathlib import Path
from IPython.display import display, Markdown
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path('/content/delaylens')] if (p/'experiments/REVIEW_PROTOCOL.md').exists()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploads = files.upload()
    archive = next(name for name in uploads if name.endswith('.zip'))
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            parts = Path(name).parts
            if not parts or parts[0] != 'delaylens' or '..' in parts or Path(name).is_absolute():
                raise ValueError('Unexpected archive member')
        z.extractall('/content')
    ROOT = Path('/content/delaylens')
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted DelayLens project.')
os.chdir(ROOT)
PYTHON = sys.executable
if IN_COLAB:
    env = Path('/content/delaylens-review-env')
    if not (env/'bin/python').exists():
        subprocess.run([sys.executable,'-m','venv',str(env)],check=True)
    PYTHON = str(env/'bin/python')
    subprocess.run([PYTHON,'-m','pip','install','-r','requirements-lock.txt'],cwd=ROOT,check=True)
def run_python(source):
    result = subprocess.run([PYTHON,'-c',source],cwd=ROOT,text=True,capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
print('Ready. This notebook reads real saved evidence; it does not refit candidates.')


Ready. This notebook reads real saved evidence; it does not refit candidates.


## Read the evidence first
The research decision and the historical API model have different roles. No production, causal-impact or savings claim is made.

In [2]:
display(Markdown((ROOT/'reports/review/SUMMARY.md').read_text()))

# ML did not pass the missing-outcome robustness gate; simple-rule performance varies by period.

Preferred development method: **route_duration_rule**. ML promoted: **No**.
Selection was frozen before June/July/August diagnostics. Both simple rules are
reported; July–August never chooses the preferred method.

## Development totals, confirmed late orders found

| Method | Hits @10 | Hits @25 | Hits @50 |
|---|---|---|---|
| enriched_boosting | 474 | 1085 | 1939 |
| monotone_enriched_boosting | 507 | 1110 | 1970 |
| original_features_boosting | 388 | 932 | 1748 |
| route_duration_rule | 408 | 893 | 1620 |
| short_promise_rule | 229 | 473 | 1003 |

## Paired selection evidence against route_duration_rule

| Model | Mean monthly gain | Months won | 95% paired CI | Worst-case simultaneous CI |
|---|---|---|---|---|
| original_features_boosting | 8.53 | 9 | [3.00, 14.47] | [-15.02, 0.36] |
| enriched_boosting | 21.27 | 11 | [15.33, 27.33] | [-3.36, 12.87] |
| monotone_enriched_boosting | 23.33 | 11 | [17.33, 29.40] | [-1.40, 15.22] |

These are mean monthly differences at the primary review budget. The ordinary
intervals describe fixed-score observed-label variation. The final column also
allows unobserved outcomes to favour the rule, with a simultaneous adjustment
for the prespecified model/rule comparisons. It is not a prediction interval for
new months. Passing the ordinary observed-label gate alone does not establish
robustness to missing outcomes.

## Retrospective stress results

| Method | Budget | Reviewed | Review share | Confirmed late hits | Selected unknown | Known-hit lift |
|---|---|---|---|---|---|---|
| short_promise_rule | 10 | 569 | 4.53% | 80 | 32 | 3.40 |
| short_promise_rule | 25 | 1397 | 11.13% | 141 | 49 | 2.46 |
| short_promise_rule | 50 | 2772 | 22.08% | 250 | 87 | 2.20 |
| route_duration_rule | 10 | 569 | 4.53% | 60 | 32 | 2.55 |
| route_duration_rule | 25 | 1397 | 11.13% | 139 | 49 | 2.42 |
| route_duration_rule | 50 | 2772 | 22.08% | 241 | 88 | 2.12 |
| frozen_original_ml | 10 | 569 | 4.53% | 53 | 21 | 2.25 |
| frozen_original_ml | 25 | 1397 | 11.13% | 93 | 45 | 1.62 |
| frozen_original_ml | 50 | 2772 | 22.08% | 161 | 80 | 1.41 |

Unknown outcomes stay in every queue. Known-hit lift compares confirmed late hits
to random review over exactly the same full cohort; it is not full-population
recall. Budgets are hypothetical capacities, not a staffing recommendation.
The frozen API artifact is a historical comparator, not the new selected method.


## Exercise the API
This sends a real JSON request through the FastAPI application. It uses an invented example, not a customer record.

In [3]:
run_python("from fastapi.testclient import TestClient\nfrom app.main import create_app\nimport json\nwith TestClient(create_app()) as client:\n    r=client.post('/predict',json=json.load(open('examples/single_order.json')))\n    r.raise_for_status()\n    print(json.dumps(r.json(),indent=2))")

{
  "order_id": "DEMO-1001",
  "delay_probability": 0.271671327417174,
  "risk_band": "high",
  "model_version": "delaylens-1.0-a967c698-94eced67",
  "release_status": "research_only",
  "warnings": []
}



## Optional browser dashboard
The Colab browser proxy requires a live signed-in Colab runtime. It was not verified by the authoring runtime. It is temporary hosting, not production deployment.

In [4]:
OPEN_DASHBOARD = False
if OPEN_DASHBOARD:
    import time
    server = subprocess.Popen([PYTHON,'-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8765'],cwd=ROOT)
    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(8765)
    else:
        print('http://127.0.0.1:8765')
else:
    print('Optional dashboard server not started. API request above was executed.')

Optional dashboard server not started. API request above was executed.


## Rebuild from raw data
Run `python scripts/rebuild.py` from the project folder using Python 3.12. It creates a new isolated checkout, installs the lock, downloads/verifies raw data, commits the fixed protocol, runs it once, then produces reports and checks. Do not alter settings to chase July–August results. The supplied results remain intact.